In [1]:
import pandas as pd
import numpy as np

from datetime import datetime, timedelta

# If you need to import your own modules later (patterns, etc.) you can do:
# import sys
# sys.path.append(r"C:\Users\admin\Desktop\Forex_algo\code")


In [2]:
DATA_PATH = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet"

df = pd.read_parquet(DATA_PATH)
df.head(), df.tail(), (df.time.min(), df.time.max(), df.shape)

(                       time  volume    mid_o    mid_h    mid_l    mid_c  \
 0 2016-01-07 00:00:00+00:00     542  1.07764  1.07832  1.07744  1.07778   
 1 2016-01-07 01:00:00+00:00    3167  1.07776  1.08100  1.07748  1.08029   
 2 2016-01-07 02:00:00+00:00    1567  1.08026  1.08176  1.07996  1.08152   
 3 2016-01-07 03:00:00+00:00     914  1.08156  1.08257  1.08150  1.08187   
 4 2016-01-07 04:00:00+00:00     649  1.08190  1.08256  1.08156  1.08236   
 
      bid_o    bid_h    bid_l    bid_c    ask_o    ask_h    ask_l    ask_c  
 0  1.07757  1.07823  1.07735  1.07770  1.07772  1.07840  1.07752  1.07787  
 1  1.07768  1.08092  1.07740  1.08020  1.07784  1.08109  1.07756  1.08038  
 2  1.08018  1.08169  1.07987  1.08144  1.08035  1.08184  1.08005  1.08159  
 3  1.08147  1.08249  1.08142  1.08178  1.08164  1.08265  1.08157  1.08196  
 4  1.08182  1.08247  1.08147  1.08228  1.08199  1.08264  1.08163  1.08245  ,
                            time  volume    mid_o    mid_h    mid_l    mid_c  \

In [3]:
df_feat = df.copy()


In [4]:
# Simple returns over different lookbacks
df_feat["ret_1"]  = df_feat["mid_c"].pct_change(1)   # 1-hour return
df_feat["ret_3"]  = df_feat["mid_c"].pct_change(3)   # 3-hour return
df_feat["ret_6"]  = df_feat["mid_c"].pct_change(6)   # 6-hour return
df_feat["ret_12"] = df_feat["mid_c"].pct_change(12)  # 12-hour return
df_feat["ret_24"] = df_feat["mid_c"].pct_change(24)  # 1-day (24h) return


In [5]:
df_feat["ma_10"] = df_feat["mid_c"].rolling(10).mean()   # ~10 hours
df_feat["ma_20"] = df_feat["mid_c"].rolling(20).mean()   # ~1 day
df_feat["ma_50"] = df_feat["mid_c"].rolling(50).mean()   # ~2 days

df_feat["dist_ma10"] = df_feat["mid_c"] - df_feat["ma_10"]
df_feat["dist_ma20"] = df_feat["mid_c"] - df_feat["ma_20"]
df_feat["dist_ma50"] = df_feat["mid_c"] - df_feat["ma_50"]


In [6]:
# Rolling volatility of price (standard deviation of close)
df_feat["std_10"] = df_feat["mid_c"].rolling(10).std()
df_feat["std_20"] = df_feat["mid_c"].rolling(20).std()

# Rolling volatility of returns
df_feat["vol_10"] = df_feat["ret_1"].rolling(10).std()
df_feat["vol_20"] = df_feat["ret_1"].rolling(20).std()


In [7]:
# Candle body & wicks on mid-price
df_feat["body"]      = df_feat["mid_c"] - df_feat["mid_o"]
df_feat["range_hl"]  = df_feat["mid_h"] - df_feat["mid_l"]
df_feat["upper_wick"] = df_feat["mid_h"] - df_feat[["mid_c", "mid_o"]].max(axis=1)
df_feat["lower_wick"] = df_feat[["mid_c", "mid_o"]].min(axis=1) - df_feat["mid_l"]

# Normalize by price to make scale-invariant
df_feat["body_rel"]      = df_feat["body"] / df_feat["mid_c"]
df_feat["range_rel"]     = df_feat["range_hl"] / df_feat["mid_c"]
df_feat["upper_wick_rel"] = df_feat["upper_wick"] / df_feat["mid_c"]
df_feat["lower_wick_rel"] = df_feat["lower_wick"] / df_feat["mid_c"]


In [8]:
df_feat["vol_chg_1"] = df_feat["volume"].pct_change(1)
df_feat["vol_chg_3"] = df_feat["volume"].pct_change(3)

# Rolling avg volume
df_feat["vol_ma_10"] = df_feat["volume"].rolling(10).mean()
df_feat["vol_ma_20"] = df_feat["volume"].rolling(20).mean()

# Volume relative to recent average
df_feat["vol_rel_10"] = df_feat["volume"] / df_feat["vol_ma_10"]
df_feat["vol_rel_20"] = df_feat["volume"] / df_feat["vol_ma_20"]


In [9]:
HORIZON = 6  # hours into the future

df_feat["future_mid_c"] = df_feat["mid_c"].shift(-HORIZON)
df_feat["future_ret"] = (df_feat["future_mid_c"] - df_feat["mid_c"]) / df_feat["mid_c"]

# Direction label: 1 = Up, 0 = Down or flat
df_feat["target"] = (df_feat["future_ret"] > 0).astype(int)


In [10]:
df_feat = df_feat.dropna().reset_index(drop=True)
df_feat.shape


(61418, 46)

In [11]:
# If you used patterns:
pattern_cols = []

# If patterns not available, you can set: pattern_cols = []

feature_cols = [
    # Returns
    "ret_1", "ret_3", "ret_6", "ret_12", "ret_24",
    
    # MAs & distance to MAs
    "ma_10", "ma_20", "ma_50",
    "dist_ma10", "dist_ma20", "dist_ma50",
    
    # Volatility
    "std_10", "std_20",
    "vol_10", "vol_20",
    
    # Candle shape (price action)
    "body_rel", "range_rel", "upper_wick_rel", "lower_wick_rel",
    
    # Volume
    "vol_chg_1", "vol_chg_3",
    "vol_ma_10", "vol_ma_20", "vol_rel_10", "vol_rel_20",
] + pattern_cols  # if using patterns


In [12]:
X = df_feat[feature_cols].astype(float)
y = df_feat["target"].astype(int)

X.shape, y.shape, y.value_counts(normalize=True)


((61418, 25),
 (61418,),
 target
 1    0.50337
 0    0.49663
 Name: proportion, dtype: float64)

In [13]:
train_size = int(len(X) * 0.8)  # 80% train, 20% validation

X_train = X.iloc[:train_size].copy()
y_train = y.iloc[:train_size].copy()
X_val   = X.iloc[train_size:].copy()
y_val   = y.iloc[train_size:].copy()

X_train.shape, X_val.shape


((49134, 25), (12284, 25))

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)


In [15]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


In [16]:
import tensorflow as tf
print(tf.__version__)

2.17.1


In [17]:
tf.config.list_physical_devices('GPU')


[]

In [ ]:
import sys
print(sys.version)


In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


PyTorch version: 2.5.1+cu121
CUDA available: True
Device name: NVIDIA GeForce RTX 4070 Laptop GPU
